In [1]:
import os
import numpy as np
import pandas as pd
import librosa
from scipy.stats import kurtosis
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib

## Parameters

In [94]:
# ── Audio framing ─────────────────────────────────────────────────
FRAME_SIZE   = 0.025   # seconds  
HOP_SIZE     = 0.01    # seconds  
N_FFT        = 1024
N_MFCC       = 13

# ── Crackle-specific frequency bands ──────────────────────────────
# Crackles have energy mainly in 200–2000 Hz (fine crackles go higher)
CRACKLE_LOW  = 200     # Hz
CRACKLE_HIGH = 2000    # Hz

# ── Post-processing ───────────────────────────────────────────────
# Crackles are short (≥ 5 ms typical); keep min duration longer than
# a single frame but shorter than the Rhonchi minimum.
MIN_DURATION_SEC  = 0.3   # seconds
MAX_GAP_FRAMES    = 30      # bridge tiny gaps between predicted bursts

In [95]:
def extract_frame_features(y, sr):
    frame_length = int(FRAME_SIZE * sr)
    hop_length   = int(HOP_SIZE  * sr)

    S = np.abs(librosa.stft(
        y,
        n_fft=N_FFT,
        hop_length=hop_length,
        win_length=frame_length
    ))
    freqs = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)

    # ── Band energy ratio (200–2000 Hz) ──────────────────────────────
    crack_mask   = (freqs >= CRACKLE_LOW) & (freqs <= CRACKLE_HIGH)
    total_energy = np.sum(S, axis=0) + 1e-8
    crack_energy = np.sum(S[crack_mask, :], axis=0)
    band_ratio   = crack_energy / total_energy

    # ── Spectral features ─────────────────────────────────────────────
    centroid = librosa.feature.spectral_centroid(S=S, sr=sr)[0]
    flatness = librosa.feature.spectral_flatness(S=S)[0]
    rolloff  = librosa.feature.spectral_rolloff(S=S, sr=sr)[0]

    zcr = librosa.feature.zero_crossing_rate(
        y, frame_length=frame_length, hop_length=hop_length
    )[0]

    psd     = S / total_energy
    entropy = -np.sum(psd * np.log(psd + 1e-8), axis=0)

    rms = librosa.feature.rms(
        y=y, frame_length=frame_length, hop_length=hop_length
    )[0]

    contrast = librosa.feature.spectral_contrast(
        S=S, sr=sr, n_fft=N_FFT, hop_length=hop_length,
        fmin=10.0, n_bands=4
    )

    # ── Impulse features ──────────────────────────────────────────────
    n_frames   = S.shape[1]
    frame_kurt = np.zeros(n_frames)
    frame_peak = np.zeros(n_frames)
    for i in range(n_frames):
        start = i * hop_length
        end   = start + frame_length
        chunk = y[start:end] if end <= len(y) else y[start:]
        if len(chunk) > 1:
            frame_kurt[i] = kurtosis(chunk)
            rms_val        = np.sqrt(np.mean(chunk ** 2)) + 1e-8
            frame_peak[i]  = np.max(np.abs(chunk)) / rms_val

    rms_slope = np.diff(rms, prepend=rms[0])

    rms_slope2 = np.diff(rms_slope, prepend=rms_slope[0])  # second derivative

    # Onset strength: positive part of second derivative only
    # (we only care about sudden rises, not falls)
    onset_strength = np.maximum(rms_slope2, 0)

    # Normalize by local RMS to make it amplitude-independent
    onset_strength_norm = onset_strength / (rms + 1e-8)

    # ── Trim all arrays to n_frames ───────────────────────────────────
    def _trim(arr):
        return arr[:n_frames] if arr.ndim == 1 else arr[:, :n_frames]

    band_ratio = _trim(band_ratio)
    centroid   = _trim(centroid)
    flatness   = _trim(flatness)
    rolloff    = _trim(rolloff)
    zcr        = _trim(zcr)
    entropy    = _trim(entropy)
    rms        = _trim(rms)
    contrast   = _trim(contrast)
    frame_kurt = _trim(frame_kurt)
    frame_peak = _trim(frame_peak)
    rms_slope  = _trim(rms_slope)
    rms_slope2          = _trim(rms_slope2)
    onset_strength      = _trim(onset_strength)
    onset_strength_norm = _trim(onset_strength_norm)

    features = np.vstack([
        band_ratio,
        centroid,
        flatness,
        rolloff,
        zcr,
        entropy,
        rms,
        contrast,
        frame_kurt,
        frame_peak,
        rms_slope,
        rms_slope2,
        onset_strength,
        onset_strength_norm
    ]).T

    return features, hop_length


def _build_column_names():
    cols  = ["band_ratio", "centroid", "flatness", "rolloff",
             "zcr", "entropy", "rms"]
    cols += [f"contrast_{i+1}" for i in range(5)]
    cols += ["kurtosis", "peak_to_rms", "rms_slope"]
    cols += ["rms_slope2", "onset_strength", "onset_strength_norm"]
    return cols

FEATURE_COLUMNS = _build_column_names()
print(f"Total features per frame: {len(FEATURE_COLUMNS)}")

Total features per frame: 18


In [96]:
def create_crackle_labels(label_file, total_frames, sr, hop_length):
    """
    Read an Audacity-style tab-separated label file.
    Expected columns: start(s) | end(s) | label

    A frame is labelled 1 (crackle) only when it is covered by
    at least one 'I' interval AND at least one 'D' interval
    simultaneously.

    Parameters
    ----------
    label_file   : path to the .txt annotation file
    total_frames : number of frames produced by the STFT
    sr           : sample rate
    hop_length   : hop length in samples

    Returns
    -------
    labels : ndarray of int, shape (total_frames,), values 0 or 1
    """
    df = pd.read_csv(
        label_file,
        sep="\t",
        header=None,
        names=["start", "end", "label"]
    )
    df["label"] = df["label"].astype(str).str.strip()

    # Build per-frame boolean masks for each label type
    mask_I = np.zeros(total_frames, dtype=bool)
    mask_D = np.zeros(total_frames, dtype=bool)

    for _, row in df.iterrows():
        start_f = max(0, int(row["start"] * sr / hop_length))
        end_f   = min(total_frames, int(row["end"]   * sr / hop_length))

        lbl = row["label"]
        if lbl == "I":
            mask_I[start_f:end_f] = True
        elif lbl == "D":
            mask_D[start_f:end_f] = True
        # Any other labels (e.g. 'E' for expiratory) are ignored

    # Crackle = overlap of I and D
    labels = (mask_I & mask_D).astype(int)

    return labels

In [97]:
def build_dataset(audio_folder):
    """
    Walk audio_folder, pair each .wav with its *_label_audacity.txt,
    extract features and crackle labels, return stacked arrays.
    """
    X_all, y_all = [], []

    audio_files = sorted(os.listdir(audio_folder))

    for file in tqdm(audio_files, desc="Processing files"):
        if not file.endswith(".wav"):
            continue

        audio_path = os.path.join(audio_folder, file)
        base_name  = file.replace(".wav", "")
        label_path = os.path.join(audio_folder, base_name + "_label_audacity.txt")

        if not os.path.exists(label_path):
            print(f"⚠  Missing label file for {file} — skipping")
            continue

        y, sr = librosa.load(audio_path, sr=None)

        features, hop_length = extract_frame_features(y, sr)

        labels = create_crackle_labels(
            label_path,
            features.shape[0],
            sr,
            hop_length
        )

        X_all.append(features)
        y_all.append(labels)

    X = np.vstack(X_all)
    y = np.hstack(y_all)
    return X, y

In [98]:
def post_process_predictions(
    predictions,
    hop_size_sec,
    min_duration_sec=MIN_DURATION_SEC,
    max_gap_frames=MAX_GAP_FRAMES
):
    """
    1. Fill small zero-gaps between positive runs (bridge gaps).
    2. Remove runs shorter than min_duration_sec.
    """
    cleaned = predictions.copy()

    # ── Pass 1: bridge tiny gaps ─────────────────────────────────────
    i = 0
    while i < len(cleaned):
        if cleaned[i] == 0:
            gap_start = i
            while i < len(cleaned) and cleaned[i] == 0:
                i += 1
            gap_end    = i
            gap_length = gap_end - gap_start

            if (
                gap_length <= max_gap_frames
                and gap_start > 0
                and gap_end < len(cleaned)
                and cleaned[gap_start - 1] == 1
                and cleaned[gap_end] == 1
            ):
                cleaned[gap_start:gap_end] = 1
        else:
            i += 1

    # ── Pass 2: remove short segments ────────────────────────────────
    min_frames = max(1, int(min_duration_sec / hop_size_sec))
    start = None

    for i in range(len(cleaned)):
        if cleaned[i] == 1 and start is None:
            start = i
        if (cleaned[i] == 0 or i == len(cleaned) - 1) and start is not None:
            end = i if cleaned[i] == 0 else i + 1
            if end - start < min_frames:
                cleaned[start:end] = 0
            start = None

    return cleaned

In [99]:
def count_crackles_from_labels(label_file):
    """Count ground-truth crackle events (I ∩ D overlaps)."""
    df = pd.read_csv(
        label_file, sep="\t", header=None,
        names=["start", "end", "label"]
    )
    df["label"] = df["label"].astype(str).str.strip()

    I_segs = df[df["label"] == "I"][["start", "end"]].values
    D_segs = df[df["label"] == "D"][["start", "end"]].values

    count = 0
    for d_start, d_end in D_segs:
        for i_start, i_end in I_segs:
            # Check temporal overlap
            if d_start < i_end and d_end > i_start:
                count += 1
                break  # count this D-segment only once

    return count


def count_crackles_from_predictions(predicted_labels):
    """Count distinct predicted crackle segments (run-length encoding)."""
    count, in_segment = 0, False
    for val in predicted_labels:
        if val == 1 and not in_segment:
            count += 1
            in_segment = True
        elif val == 0:
            in_segment = False
    return count


def evaluate_file_counts(test_csv_folder, audio_folder):
    results = []

    for csv_file in sorted(os.listdir(test_csv_folder)):
        if not csv_file.endswith("_test.csv"):
            continue

        base       = csv_file.replace("_test.csv", "")
        label_file = os.path.join(audio_folder, base + "_label_audacity.txt")

        if not os.path.exists(label_file):
            print(f"⚠  Missing label file for {base}")
            continue

        df = pd.read_csv(os.path.join(test_csv_folder, csv_file))

        true_count  = count_crackles_from_labels(label_file)
        pred_raw    = count_crackles_from_predictions(df["predicted_label_raw"].values)
        pred_clean  = count_crackles_from_predictions(df["predicted_label_clean"].values)

        results.append({
            "file":           base,
            "true_count":     true_count,
            "predicted_raw":  pred_raw,
            "predicted_clean": pred_clean,
            "diff_raw":       pred_raw   - true_count,
            "diff_clean":     pred_clean - true_count,
        })

        print(f"{base}")
        print(f"  True crackles  : {true_count}")
        print(f"  Predicted      : {pred_clean}  (diff: {pred_clean - true_count:+d})")
        print()

    results_df = pd.DataFrame(results)
    results_df.to_csv("crackle_count_comparison.csv", index=False)

    print("=" * 50)
    print(f"Files evaluated      : {len(results)}")
    print(f"Total true count     : {results_df['true_count'].sum()}")
    print(f"Total predicted      : {results_df['predicted_clean'].sum()}")
    print(f"Mean abs error       : {results_df['diff_clean'].abs().mean():.2f}")
    print("\n✅ Saved → crackle_count_comparison.csv")


In [100]:
def main():
    audio_folder  = r"E:\Research Project (Prof. Preeti Rao)\Top files by Crackle count"
    output_folder = "output"
    os.makedirs(output_folder + "/train", exist_ok=True)
    os.makedirs(output_folder + "/test",  exist_ok=True)

    audio_files = sorted([f for f in os.listdir(audio_folder) if f.endswith(".wav")])
    train_files, test_files = train_test_split(audio_files, test_size=0.2, random_state=42)

    X_train_all, y_train_all = [], []

    for file in tqdm(train_files, desc="Train"):
        base       = file.replace(".wav", "")
        label_path = os.path.join(audio_folder, base + "_label_audacity.txt")
        if not os.path.exists(label_path):
            print(f"⚠  Missing label for {file}")
            continue
        y, sr = librosa.load(os.path.join(audio_folder, file), sr=None)
        features, hop_length = extract_frame_features(y, sr)
        labels = create_crackle_labels(label_path, features.shape[0], sr, hop_length)
        X_train_all.append(features)
        y_train_all.append(labels)
        df = pd.DataFrame(features, columns=FEATURE_COLUMNS)
        df["true_label"] = labels
        df.to_csv(f"{output_folder}/train/{base}_train.csv", index=False)

    X_train = np.vstack(X_train_all)
    y_train = np.hstack(y_train_all)

    scaler         = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    # ── Onset bias in sample weights ─────────────────────────────────
    onset_col  = FEATURE_COLUMNS.index("onset_strength_norm")
    onset_vals = X_train_scaled[:, onset_col]

    class_counts  = np.bincount(y_train.astype(int))
    base_weight   = len(y_train) / (2 * class_counts)
    sample_weight = np.where(
        y_train == 1,
        base_weight[1] * (1 + onset_vals),
        base_weight[0]
    )

    clf = RandomForestClassifier(
        n_estimators=300,
        n_jobs=-1,
        random_state=42
    )

    print("\nTraining …")
    clf.fit(X_train_scaled, y_train, sample_weight=sample_weight)

    y_test_all, y_pred_all, y_pred_clean_all = [], [], []

    for file in tqdm(test_files, desc="Test"):
        base       = file.replace(".wav", "")
        label_path = os.path.join(audio_folder, base + "_label_audacity.txt")
        if not os.path.exists(label_path):
            print(f"⚠  Missing label for {file}")
            continue
        y, sr = librosa.load(os.path.join(audio_folder, file), sr=None)
        features, hop_length = extract_frame_features(y, sr)
        labels = create_crackle_labels(label_path, features.shape[0], sr, hop_length)

        X_test_scaled = scaler.transform(features)
        y_pred        = clf.predict(X_test_scaled)
        y_pred_clean  = post_process_predictions(y_pred, hop_size_sec=HOP_SIZE)

        y_test_all.append(labels)
        y_pred_all.append(y_pred)
        y_pred_clean_all.append(y_pred_clean)

        df = pd.DataFrame(features, columns=FEATURE_COLUMNS)
        df["true_label"]            = labels
        df["predicted_label_raw"]   = y_pred
        df["predicted_label_clean"] = y_pred_clean
        df.to_csv(f"{output_folder}/test/{base}_test.csv", index=False)

    y_test_all       = np.hstack(y_test_all)
    y_pred_all       = np.hstack(y_pred_all)
    y_pred_clean_all = np.hstack(y_pred_clean_all)

    print("\n" + "="*55)
    print("PERFORMANCE METRICS (raw predictions)")
    print("="*55)
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test_all, y_pred_all))
    print("\nClassification Report:")
    print(classification_report(y_test_all, y_pred_all,
                                target_names=["No Crackle", "Crackle"]))

    print("\n" + "="*55)
    print("PERFORMANCE METRICS (post-processed predictions)")
    print("="*55)
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test_all, y_pred_clean_all))
    print("\nClassification Report:")
    print(classification_report(y_test_all, y_pred_clean_all,
                                target_names=["No Crackle", "Crackle"]))

    joblib.dump(clf,    "crackle_model.pkl")
    joblib.dump(scaler, "crackle_scaler.pkl")
    print("\n✅ Done.")

main()

audio_folder = r"E:\Research Project (Prof. Preeti Rao)\Top files by Crackle count"
evaluate_file_counts("output/test", audio_folder)

Train:  89%|████████▉ | 82/92 [01:04<00:07,  1.42it/s]C:\Users\HP\AppData\Local\Temp\ipykernel_18176\3783857114.py:49: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  frame_kurt[i] = kurtosis(chunk)
Train: 100%|██████████| 92/92 [01:11<00:00,  1.28it/s]



Training …


Test: 100%|██████████| 23/23 [00:19<00:00,  1.20it/s]



PERFORMANCE METRICS (raw predictions)

Confusion Matrix:
[[20812  2261]
 [ 9119  2331]]

Classification Report:
              precision    recall  f1-score   support

  No Crackle       0.70      0.90      0.79     23073
     Crackle       0.51      0.20      0.29     11450

    accuracy                           0.67     34523
   macro avg       0.60      0.55      0.54     34523
weighted avg       0.63      0.67      0.62     34523


PERFORMANCE METRICS (post-processed predictions)

Confusion Matrix:
[[14230  8843]
 [ 3581  7869]]

Classification Report:
              precision    recall  f1-score   support

  No Crackle       0.80      0.62      0.70     23073
     Crackle       0.47      0.69      0.56     11450

    accuracy                           0.64     34523
   macro avg       0.63      0.65      0.63     34523
weighted avg       0.69      0.64      0.65     34523


✅ Done.
steth_20181114_09_56_52
  True crackles  : 8
  Predicted      : 8  (diff: +0)

steth_20190516_14_24_